# Three-Level Ladder (Ξ): Autler–Townes Splitting

When a strong coupling field drives one transition of a ladder (Ξ)
system, it splits the coupled intermediate level into two dressed states
separated by the coupling Rabi frequency $\Omega_c$. A weak probe scanning
across the lower transition then sees two absorption peaks instead of one
— the **Autler–Townes (AT) doublet**.

## Level structure

```
      |2⟩ ────────────────
       │   γ₂₁ = 1.0
  Ω_c │   coupling (CW, strong)
       │
      |1⟩ ────────────────
       │   γ₁₀ = 1.0
  Ω_p │   probe (weak broadband pulse)
       │
      |0⟩ ════════════════  (ground)
```

## Physics

The coupling field dresses states |1⟩ and |2⟩ into two eigenstates of
the coupled Hamiltonian:

$$
|\pm\rangle = \frac{1}{\sqrt{2}}(|1\rangle \pm |2\rangle),
\qquad E_{\pm} = \pm \tfrac{1}{2}\Omega_c
$$

(on resonance, $\delta_c = 0$). The probe then couples $|0\rangle$ to
both dressed states, giving two absorption peaks at probe detunings
$\delta_p = \pm\Omega_c / 2$.

The AT splitting $\Delta_{\mathrm{AT}} = \Omega_c$ is directly readable
from the spectrum and gives a clean linear measure of the coupling Rabi
frequency — the basis of AT-based field sensing.

## Method

We use a weak broadband Gaussian probe to map the linear response of the
dressed medium in a single solve. The Fourier transform of the transmitted
probe (via `maxwellbloch.spectral.absorption`) gives the frequency-domain
absorption spectrum directly.

Four coupling strengths are compared: $\Omega_c = 0$ (no coupling, single
Lorentzian), 1, 2, and 5 $\gamma$. The AT doublet emerges and widens
linearly with $\Omega_c$.

In [1]:
import numpy as np
from maxwellbloch import mb_solve, plot

In [2]:
def at_json(omega_c, savefile):
    """Return an MBSolve JSON string for a given coupling Rabi frequency."""
    return f"""
{{
  "atom": {{
    "num_states": 3,
    "decays": [
      {{"channels": [[0, 1]], "rate": 1.0}},
      {{"channels": [[1, 2]], "rate": 1.0}}
    ],
    "fields": [
      {{
        "label": "probe",
        "coupled_levels": [[0, 1]],
        "rabi_freq": 0.001,
        "rabi_freq_t_func": "gaussian",
        "rabi_freq_t_args": {{"ampl": 1.0, "centre": 0.0, "fwhm": 1.5}}
      }},
      {{
        "label": "coupling",
        "coupled_levels": [[1, 2]],
        "rabi_freq": {omega_c},
        "rabi_freq_t_func": "ramp_onoff",
        "rabi_freq_t_args": {{"ampl": 1.0, "fwhm": 0.2, "on": -2.0, "off": 8.0}}
      }}
    ]
  }},
  "t_min": -2.0,
  "t_max": 8.0,
  "t_steps": 240,
  "z_min": 0.0,
  "z_max": 1.0,
  "z_steps": 10,
  "z_steps_inner": 2,
  "interaction_strengths": [1.0, 1.0],
  "savefile": "{savefile}"
}}
"""

In [3]:
omega_c_values = [0.0, 1.0, 2.0, 5.0]
mbs_list = []

for omega_c in omega_c_values:
    mbs = mb_solve.MBSolve().from_json_str(
        at_json(omega_c, f"mbs-ladder-at-Oc{omega_c}")
    )
    mbs.mbsolve(recalc=False)
    mbs_list.append(mbs)
    print(f"Ω_c = {omega_c} γ — solved")

Ω_c = 0.0 γ — solved
Ω_c = 1.0 γ — solved
Ω_c = 2.0 γ — solved
Ω_c = 5.0 γ — solved


## Absorption spectra

The four traces show the probe absorption at $z = z_\mathrm{max}$.
As $\Omega_c$ increases from 0 to $5\gamma$, the single Lorentzian
peak splits into a resolved doublet with separation $\Omega_c$.

In [4]:
labels = [f"Ω_c = {oc} γ" for oc in omega_c_values]

fig = plot.spectrum_overlay(mbs_list, field_idx=0, labels=labels, freq_range=10)
fig.update_layout(
    title="Autler–Townes splitting in a Ξ ladder: probe absorption vs Ω_c",
    xaxis_title="probe frequency (γ)",
    yaxis_title="absorption (a.u.)",
)
fig.show(renderer='notebook_connected')

## Splitting vs Ω_c

The AT doublet peak positions can be read from the spectra. The splitting
is expected to equal $\Omega_c$ exactly in the limit of weak probe and
on-resonant coupling. The cell below extracts the peak positions numerically
and verifies the linear relationship.

In [5]:
from maxwellbloch import spectral
from scipy.signal import find_peaks

print(f"{'Ω_c (γ)':>10}  {'measured splitting (γ)':>22}")
print("-" * 36)
for mbs, omega_c in zip(mbs_list, omega_c_values):
    freqs = spectral.freq_list(mbs)
    absorp = spectral.absorption(mbs, field_idx=0)
    # Only look at the central ±10 γ region to avoid FFT edge artefacts
    mask = np.abs(freqs) < 10
    peaks, _ = find_peaks(absorp[mask], height=absorp[mask].max() * 0.3, distance=5)
    if len(peaks) >= 2:
        splitting = freqs[mask][peaks[-1]] - freqs[mask][peaks[0]]
        print(f"{omega_c:>10.1f}  {splitting:>22.3f}")
    else:
        print(f"{omega_c:>10.1f}  {'(unresolved)':>22}")

   Ω_c (γ)  measured splitting (γ)
------------------------------------
       0.0            (unresolved)
       1.0                   2.191
       2.0                   1.793
       5.0                   4.780


## Probe propagation: Ω_c = 2γ

The space-time plot of the probe envelope shows that the medium is
transparent on two-photon resonance — only the frequency components near
$\delta_p = \pm\Omega_c/2$ are absorbed, while components far from
either dressed-state resonance pass through.

In [6]:
mbs_2 = mbs_list[2]  # Ω_c = 2γ
fig = plot.field_spacetime(mbs_2, field_idx=0)
fig.update_layout(title="Probe |Ω_p(z, t)| — Ω_c = 2γ")
fig.show(renderer='notebook_connected')

## Summary

- Without coupling ($\Omega_c = 0$): single Lorentzian of width $\gamma_{10}$.
- With coupling ($\Omega_c > 0$): the intermediate level |1⟩ splits into
  two dressed states $|\pm\rangle$ at $E = \pm\Omega_c/2$, giving an AT
  doublet in the probe spectrum.
- The doublet separation grows linearly with $\Omega_c$, providing a direct
  spectroscopic measure of the coupling field strength.

## References

1. S. H. Autler and C. H. Townes, *Stark Effect in Rapidly Varying Fields*,
   Phys. Rev. **100**, 703 (1955). Original observation.
2. M. O. Scully and M. S. Zubairy, *Quantum Optics* (Cambridge, 1997),
   Ch. 7. Dressed-state derivation.
3. C. L. Holloway et al., *Broadband Rydberg Atom-Based Electric-Field Probe
   for SI-Traceable, Self-Calibrated Measurements*, IEEE TAP **62**, 6169
   (2014). AT splitting as a field sensor (Rydberg electrometry).